# PopOut: The Game

### Introdução
Este trabalho foi realizado no âmbito da Unidade Curricular Inteligência Artificial (CC2006) com o objetivo de criar o jogo PopOut de forma a que seja possível haver jogos de humano vs humano, humano vs computador e computador vs computador. Os algoritmos implementados são o **Monte Carlo Tree Search (MCTS)** e **Decision Trees com o procedimento ID3**.

### Etapas do Trabalho
Neste projeto vamos:
1. Criar um jogo funcional, tendo em conta todas as regras dadas.
2. Implementar o algoritmo de Monte Carlo Tree Search (MCTS).
3. Implementar o algoritmo de Decision Trees com ID3.
    
    3.1. Teste usando o dataset dado "iris dataset".
    
    3.2. Teste final no jogo já criado, gerando o dataset com o MCTS.


---
## 1. Criação do Jogo

A primeira coisa que fizemos foi criar classes para organizar o código. Em Python, uma **classe** é uma forma de agrupar dados (atributos) e comportamentos (métodos) num único objeto. Por exemplo, a classe `Board` guarda o estado do tabuleiro e sabe como fazer drop, pop, verificar vitória, etc.

Optámos por 3 classes principais:
- **`Board`**: representa o tabuleiro 6×7, implementa os movimentos e as verificações de vitória
- **`Player`**: representa um jogador (humano ou IA), trata do input e da escolha de jogadas
- **`Game`**: gere o fluxo do jogo — turnos, regras especiais (empate por repetição, tabuleiro cheio, pop simultâneo)

Esta separação facilita a manutenção do código e permite ligar os algoritmos de IA (MCTS e Decision Tree) sem alterar a lógica do jogo.

### 1.1 Classe `Board`

O tabuleiro é uma matriz de 6 linhas × 7 colunas. A **linha 0 é o topo** e a **linha 5 é a base** (fundo). Os discos caem por gravidade, portanto um `drop` coloca sempre o disco na posição mais baixa disponível da coluna.

```
    0 1 2 3 4 5 6   <- colunas
0 [ - - - - - - - ]  <- topo
1 [ - - - - - - - ]
2 [ - - - - - - - ]
3 [ - - - - - - - ]
4 [ X - - O - X O ]
5 [ X - O X O X O ]  <- base
```

Decisões de implementação relevantes:
- `copy()` usa `deepcopy` para garantir que o MCTS pode simular jogadas sem alterar o tabuleiro real
- `to_tuple()` converte o grid num tuplo imutável e *hashable*, necessário para guardar o histórico de estados e detetar a regra das 3 repetições

In [ ]:
import copy
import math
import random
import time
from collections import Counter


class Board:
    """
    Representa o tabuleiro do jogo PopOut (variante do Connect-4).

    O tabuleiro tem 6 linhas e 7 colunas.
    A linha 0 é o TOPO e a linha 5 é a BASE (fundo).
    Os discos caem por gravidade, portanto a base é sempre a linha 5.
    """

    ROWS = 6
    COLS = 7
    EMPTY = '-'

    def __init__(self):
        # Criamos uma matriz 6 linhas x 7 colunas, tudo vazio
        self.grid = [[self.EMPTY] * self.COLS for _ in range(self.ROWS)]

    # ------------------------------------------------------------------
    # DISPLAY
    # ------------------------------------------------------------------

    def display(self):
        """
        Mostra o tabuleiro no terminal, no formato pedido no enunciado.
        Exemplo:
            -------
            -------
            X--O-XO
            X-OXOXO
            0123456
        """
        print()
        for row in self.grid:
            print(''.join(row))
        print(''.join(str(c) for c in range(self.COLS)))
        print()

    # ------------------------------------------------------------------
    # MOVIMENTOS
    # ------------------------------------------------------------------

    def drop(self, col, player):
        """
        Coloca um disco do jogador na coluna col (drop move).
        O disco cai até à posição mais baixa disponível (gravidade).
        Retorna True se válido, False se a coluna está cheia.
        """
        for row in range(self.ROWS - 1, -1, -1):
            if self.grid[row][col] == self.EMPTY:
                self.grid[row][col] = player
                return True
        return False

    def pop(self, col, player):
        """
        Remove o disco do jogador na base da coluna col (pop move).
        Todos os discos acima descem uma posição (por gravidade).
        Retorna True se válido, False caso contrário.
        """
        if not self.can_pop(col, player):
            return False
        # Percorre de baixo para cima e copia a linha de cima para baixo
        for row in range(self.ROWS - 1, 0, -1):
            self.grid[row][col] = self.grid[row - 1][col]
        self.grid[0][col] = self.EMPTY
        return True

    def can_pop(self, col, player):
        """Verifica se o jogador pode fazer pop na coluna col."""
        return self.grid[self.ROWS - 1][col] == player

    def get_valid_drops(self):
        """Retorna colunas onde é possível fazer drop (topo vazio)."""
        return [col for col in range(self.COLS) if self.grid[0][col] == self.EMPTY]

    def get_valid_pops(self, player):
        """Retorna colunas onde o jogador pode fazer pop."""
        return [col for col in range(self.COLS) if self.can_pop(col, player)]

    def get_all_moves(self, player):
        """
        Retorna todos os movimentos possíveis para o jogador.
        Formato: lista de tuplos ('drop'/'pop', coluna).
        """
        moves = [('drop', col) for col in self.get_valid_drops()]
        moves += [('pop', col) for col in self.get_valid_pops(player)]
        return moves

    # ------------------------------------------------------------------
    # VERIFICAÇÃO DE VITÓRIA
    # ------------------------------------------------------------------

    def check_win(self, player):
        """Verifica se o jogador tem 4 discos em linha (horizontal, vertical ou diagonal)."""
        return (
            self._check_horizontal(player) or
            self._check_vertical(player) or
            self._check_diagonal_down(player) or
            self._check_diagonal_up(player)
        )

    def _check_horizontal(self, player):
        for row in range(self.ROWS):
            for col in range(self.COLS - 3):
                if all(self.grid[row][col + k] == player for k in range(4)):
                    return True
        return False

    def _check_vertical(self, player):
        for row in range(self.ROWS - 3):
            for col in range(self.COLS):
                if all(self.grid[row + k][col] == player for k in range(4)):
                    return True
        return False

    def _check_diagonal_down(self, player):
        """Diagonal descendente (↘)."""
        for row in range(self.ROWS - 3):
            for col in range(self.COLS - 3):
                if all(self.grid[row + k][col + k] == player for k in range(4)):
                    return True
        return False

    def _check_diagonal_up(self, player):
        """Diagonal ascendente (↗)."""
        for row in range(3, self.ROWS):
            for col in range(self.COLS - 3):
                if all(self.grid[row - k][col + k] == player for k in range(4)):
                    return True
        return False

    # ------------------------------------------------------------------
    # ESTADO DO TABULEIRO
    # ------------------------------------------------------------------

    def is_full(self):
        """Retorna True se o tabuleiro estiver completamente cheio."""
        return all(self.grid[0][col] != self.EMPTY for col in range(self.COLS))

    def copy(self):
        """
        Retorna uma cópia independente do tabuleiro.
        Essencial para o MCTS simular jogadas sem alterar o tabuleiro real.
        """
        new_board = Board()
        new_board.grid = copy.deepcopy(self.grid)
        return new_board

    def to_tuple(self):
        """
        Converte o tabuleiro para um tuplo de tuplos (imutável e hashable).
        Necessário para guardar o histórico e detetar repetições.
        """
        return tuple(tuple(row) for row in self.grid)

    def to_flat_list(self):
        """
        Retorna o tabuleiro como uma lista plana de 42 valores.
        Usado para gerar features para a Decision Tree.
        """
        return [cell for row in self.grid for cell in row]


print('Classe Board definida com sucesso.')

### 1.2 Classe `Player`

A classe `Player` representa um jogador — humano ou IA. O atributo `is_human` distingue os dois casos. Quando `is_human=False`, o método `get_move` será substituído pela chamada ao algoritmo de IA correspondente (MCTS ou Decision Tree), que é passado como parâmetro `ai_func`.

In [ ]:
class Player:
    """
    Representa um jogador no jogo PopOut.

    Atributos:
        name     - nome do jogador
        symbol   - símbolo no tabuleiro ('X' ou 'O')
        is_human - True se for humano, False se for IA
        ai_func  - função de IA que recebe (board, symbol) e retorna (tipo, col)
    """

    def __init__(self, name, symbol, is_human=True, ai_func=None):
        self.name = name
        self.symbol = symbol
        self.is_human = is_human
        self.ai_func = ai_func  # função de IA, definida nas secções seguintes

    def get_move(self, board):
        """
        Obtém o próximo movimento.
        Se for humano, pede input ao utilizador.
        Se for IA, chama a função ai_func.
        Retorna um tuplo (tipo, coluna), ex: ('drop', 3) ou ('pop', 1).
        """
        if not self.is_human:
            return self.ai_func(board, self.symbol)

        # --- Input humano ---
        valid_drops = board.get_valid_drops()
        valid_pops  = board.get_valid_pops(self.symbol)

        print(f"É a vez de {self.name} ({self.symbol})")
        print(f"  Drops possíveis nas colunas: {valid_drops}")
        print(f"  Pops possíveis nas colunas:  {valid_pops}")

        while True:
            move_type = input("  Tipo de jogada (drop/pop): ").strip().lower()
            if move_type not in ('drop', 'pop'):
                print("  Escolhe 'drop' ou 'pop'.")
                continue
            if move_type == 'pop' and not valid_pops:
                print("  Não tens discos na base para fazer pop.")
                continue
            try:
                col = int(input(f"  Coluna (0-{board.COLS - 1}): "))
            except ValueError:
                print("  Introduz um número válido.")
                continue
            if move_type == 'drop' and col not in valid_drops:
                print(f"  Não podes fazer drop na coluna {col}.")
                continue
            if move_type == 'pop' and col not in valid_pops:
                print(f"  Não podes fazer pop na coluna {col}.")
                continue
            return (move_type, col)

    def __str__(self):
        return f"{self.name} ({self.symbol})"


print('Classe Player definida com sucesso.')

### 1.3 Classe `Game`

A classe `Game` gere o loop principal do jogo e implementa as **3 regras especiais do PopOut**:

1. **Pop simultâneo**: se um pop cria 4 em linha para ambos os jogadores, quem fez o pop ganha.
2. **Tabuleiro cheio**: o jogador a mover pode optar por declarar empate em vez de jogar.
3. **Repetições**: se o mesmo estado do tabuleiro se repetir 3 vezes, o jogo é empate.

O histórico de estados é guardado num dicionário `{estado: contagem}`, onde o estado é o resultado de `board.to_tuple()` — um tuplo imutável que pode ser usado como chave de dicionário.

In [ ]:
class Game:
    """
    Gere o fluxo do jogo PopOut entre dois jogadores.

    Implementa as 3 regras especiais:
      1. Pop simultâneo → quem fez o pop ganha
      2. Tabuleiro cheio → jogador pode optar por empate
      3. Repetições → após 3 repetições do mesmo estado, empate
    """

    def __init__(self, player1, player2):
        self.board = Board()
        self.players = [player1, player2]
        # Histórico: {estado_tabuleiro: nº de vezes que apareceu}
        self.history = {}
        self.current_idx = 0

    @property
    def current_player(self):
        """Retorna o jogador que tem a vez."""
        return self.players[self.current_idx]

    @property
    def other_player(self):
        """Retorna o adversário."""
        return self.players[1 - self.current_idx]

    def switch_player(self):
        self.current_idx = 1 - self.current_idx

    def record_state(self):
        """Regista o estado atual e retorna quantas vezes já apareceu."""
        state = self.board.to_tuple()
        self.history[state] = self.history.get(state, 0) + 1
        return self.history[state]

    def run(self):
        """Loop principal do jogo. Retorna o símbolo do vencedor ou 'draw'."""
        print("\n=== BEM-VINDO AO POPOUT ===")
        print(f"  {self.players[0]}  vs  {self.players[1]}")
        print("=" * 27)

        while True:
            self.board.display()
            player = self.current_player

            # --- Regra 2: Tabuleiro cheio ---
            if self.board.is_full():
                print("O tabuleiro está cheio!")
                valid_pops = self.board.get_valid_pops(player.symbol)
                if not valid_pops:
                    print("Nenhum pop disponível. Empate!")
                    return 'draw'
                if player.is_human:
                    choice = input(
                        f"{player.name}, queres fazer pop ou declarar empate? (pop/empate): "
                    ).strip().lower()
                    if choice == 'empate':
                        print("\nEmpate declarado! Tabuleiro cheio.")
                        return 'draw'
                else:
                    # IA escolhe sempre jogar (pode ser melhorado)
                    pass

            # --- Obter e aplicar jogada ---
            move_type, col = player.get_move(self.board)
            if not player.is_human:
                print(f"  {player.name} jogou: {move_type} na coluna {col}")

            result = self._apply_move(move_type, col, player)

            if result == 'win':
                self.board.display()
                winner = self.current_player  # pode ter mudado em _apply_move
                print(f"\n{winner.name} ({winner.symbol}) GANHOU!")
                return winner.symbol

            if result == 'draw':
                self.board.display()
                print("\nEmpate!")
                return 'draw'

            # --- Regra 3: Repetições ---
            count = self.record_state()
            if count >= 3:
                self.board.display()
                print(f"\nEstado repetido {count} vezes. Empate por repetição!")
                return 'draw'

            self.switch_player()

    def _apply_move(self, move_type, col, player):
        """
        Aplica o movimento e verifica o resultado.
        Retorna 'win', 'draw' ou None (jogo continua).

        Regra 1 (pop simultâneo): se ambos ficam com 4 em linha após um pop,
        quem fez o pop ganha.
        """
        if move_type == 'drop':
            self.board.drop(col, player.symbol)
            if self.board.check_win(player.symbol):
                return 'win'

        elif move_type == 'pop':
            self.board.pop(col, player.symbol)
            player_wins   = self.board.check_win(player.symbol)
            opponent_wins = self.board.check_win(self.other_player.symbol)

            if player_wins:
                # Quem fez o pop ganha sempre (mesmo com empate simultâneo)
                return 'win'
            elif opponent_wins:
                # Adversário ganhou: troca o índice para o adversário ser o vencedor
                self.current_idx = 1 - self.current_idx
                return 'win'

        return None


print('Classe Game definida com sucesso.')

### 1.4 Teste rápido do jogo (Humano vs Humano)

Para testar a lógica do tabuleiro sem correr o jogo interativo, vamos simular algumas jogadas manualmente.

In [ ]:
# Teste da classe Board
b = Board()

# Simular algumas jogadas
for col in [3, 3, 3, 4, 4, 4, 5, 5, 5]:
    b.drop(col, 'X' if col in [3, 5] else 'O')

b.drop(6, 'X')
b.display()

print('Vitória de X:', b.check_win('X'))
print('Vitória de O:', b.check_win('O'))
print('Drops válidos:', b.get_valid_drops())
print('Pops válidos para X:', b.get_valid_pops('X'))
print('Todos os movimentos de X:', b.get_all_moves('X'))

In [ ]:
# Para jogar Humano vs Humano, descomentar a célula abaixo:

# p1 = Player('Jogador 1', 'X', is_human=True)
# p2 = Player('Jogador 2', 'O', is_human=True)
# game = Game(p1, p2)
# result = game.run()
# print('Resultado:', result)

---
## 2. Monte Carlo Tree Search (MCTS)

O **Monte Carlo Tree Search** é um algoritmo de pesquisa adversarial baseado em simulações aleatórias. Em vez de explorar toda a árvore de jogo (como o Minimax), o MCTS faz simulações (*playouts*) até ao fim do jogo para estimar o valor de cada jogada.

### As 4 fases do MCTS

1. **Seleção**: a partir da raiz, desce na árvore escolhendo o nó filho com maior UCT até chegar a um nó não totalmente expandido
2. **Expansão**: adiciona um novo nó filho ao nó selecionado
3. **Simulação** (*rollout*): a partir do novo nó, simula o jogo até ao fim com jogadas aleatórias
4. **Retropropagação**: propaga o resultado da simulação de volta até à raiz, atualizando visitas e vitórias

### UCT (Upper Confidence Bound for Trees)

A fórmula UCT equilibra exploração e exploração:

$$UCT = \frac{w_i}{n_i} + C \cdot \sqrt{\frac{\ln N}{n_i}}$$

Onde:
- $w_i$ = número de vitórias no nó $i$
- $n_i$ = número de visitas ao nó $i$
- $N$ = número de visitas ao nó pai
- $C$ = constante de exploração (tipicamente $\sqrt{2}$)

Nós não visitados têm UCT = ∞, garantindo que são explorados primeiro.

In [ ]:
class MCTSNode:
    """
    Nó da árvore MCTS.

    Cada nó representa um estado do jogo após uma jogada.
    Guarda:
      - o tabuleiro nesse estado (cópia independente)
      - o jogador que acabou de jogar (para saber de quem é a vez)
      - a jogada que levou a este estado
      - estatísticas: visitas e vitórias
      - filhos expandidos e jogadas ainda não exploradas
    """

    def __init__(self, board, player, move=None, parent=None):
        self.board   = board    # estado do tabuleiro neste nó
        self.player  = player   # jogador que jogou para chegar a este estado
        self.move    = move     # (tipo, col) que gerou este nó
        self.parent  = parent
        self.children = []

        self.visits = 0
        self.wins   = 0

        # Jogadas ainda não exploradas a partir deste estado
        # O adversário é quem vai jogar a seguir
        opponent = 'O' if player == 'X' else 'X'
        self.untried_moves = board.get_all_moves(opponent)

    def uct_score(self, c=math.sqrt(2)):
        """
        Calcula o score UCT deste nó.
        Nós não visitados retornam infinito para garantir exploração.
        """
        if self.visits == 0:
            return float('inf')
        exploitation = self.wins / self.visits
        exploration  = c * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploitation + exploration

    def is_fully_expanded(self):
        """True se todos os filhos possíveis já foram adicionados."""
        return len(self.untried_moves) == 0

    def best_child(self, c=math.sqrt(2)):
        """Retorna o filho com maior score UCT."""
        return max(self.children, key=lambda child: child.uct_score(c))


def mcts(board, player, n_simulations=500, c=math.sqrt(2)):
    """
    Algoritmo MCTS principal.

    Parâmetros:
        board         - estado atual do tabuleiro
        player        - símbolo do jogador que vai jogar ('X' ou 'O')
        n_simulations - número de simulações a correr
        c             - constante de exploração UCT

    Retorna o melhor movimento encontrado: (tipo, coluna)
    """
    opponent = 'O' if player == 'X' else 'X'

    # Raiz: estado atual, o 'player' anterior foi o adversário
    # (para que a raiz expanda os movimentos do 'player' atual)
    root = MCTSNode(board.copy(), opponent)

    for _ in range(n_simulations):
        node = root

        # 1. SELEÇÃO: desce na árvore pelo UCT
        while node.is_fully_expanded() and node.children:
            node = node.best_child(c)

        # 2. EXPANSÃO: adiciona um filho não explorado
        if node.untried_moves:
            move = random.choice(node.untried_moves)
            node.untried_moves.remove(move)

            new_board = node.board.copy()
            next_player = 'O' if node.player == 'X' else 'X'
            if move[0] == 'drop':
                new_board.drop(move[1], next_player)
            else:
                new_board.pop(move[1], next_player)

            child = MCTSNode(new_board, next_player, move=move, parent=node)
            node.children.append(child)
            node = child

        # 3. SIMULAÇÃO (rollout): joga aleatoriamente até ao fim
        result = _rollout(node.board.copy(), node.player)

        # 4. RETROPROPAGAÇÃO: atualiza visitas e vitórias
        _backpropagate(node, result, player)

    # Escolhe o filho mais visitado (mais robusto que o de maior taxa de vitórias)
    best = max(root.children, key=lambda c: c.visits)
    return best.move


def _rollout(board, last_player):
    """
    Simula o jogo até ao fim com jogadas aleatórias.
    Retorna o símbolo do vencedor, ou 'draw'.
    """
    current = 'O' if last_player == 'X' else 'X'
    history = {}

    for _ in range(200):  # limite de segurança para evitar loops infinitos
        # Verifica se o último a jogar ganhou
        if board.check_win(last_player):
            return last_player

        moves = board.get_all_moves(current)
        if not moves:
            return 'draw'

        if board.is_full():
            return 'draw'

        move = random.choice(moves)
        if move[0] == 'drop':
            board.drop(move[1], current)
        else:
            board.pop(move[1], current)

        # Regra das repetições (simplificada no rollout)
        state = board.to_tuple()
        history[state] = history.get(state, 0) + 1
        if history[state] >= 3:
            return 'draw'

        last_player, current = current, last_player

    return 'draw'


def _backpropagate(node, result, root_player):
    """
    Propaga o resultado da simulação de volta até à raiz.
    Incrementa visitas em todos os nós, e vitórias nos nós
    cujo jogador coincide com o vencedor.
    """
    while node is not None:
        node.visits += 1
        if result == root_player:
            node.wins += 1
        node = node.parent


def mcts_move(board, player, n_simulations=500):
    """
    Função auxiliar compatível com a interface do Player.
    Recebe (board, player_symbol) e retorna (tipo, coluna).
    """
    return mcts(board, player, n_simulations=n_simulations)


print('MCTS definido com sucesso.')

### 2.1 Teste do MCTS

Vamos testar o MCTS num tabuleiro com um estado já avançado, variando o número de simulações para comparar a qualidade das jogadas escolhidas.

In [ ]:
# Teste: MCTS num tabuleiro semi-preenchido
test_board = Board()
for col in [3, 4, 3, 4, 3, 4]:  # X quase a ganhar na coluna 3
    test_board.drop(col, 'X' if col == 3 else 'O')

print('Estado do tabuleiro de teste:')
test_board.display()

print('Testando MCTS com diferentes números de simulações:')
for n_sims in [100, 300, 500, 1000]:
    start = time.time()
    move = mcts_move(test_board, 'X', n_simulations=n_sims)
    elapsed = time.time() - start
    print(f'  n_simulations={n_sims:4d} → jogada: {move}  ({elapsed:.2f}s)')

In [ ]:
# Para jogar Humano vs MCTS, descomentar:

# p1 = Player('Humano', 'X', is_human=True)
# p2 = Player('MCTS', 'O', is_human=False, ai_func=lambda b, s: mcts_move(b, s, n_simulations=300))
# game = Game(p1, p2)
# result = game.run()

---
## 3. Decision Trees com ID3

O algoritmo **ID3** (Iterative Dichotomiser 3) constrói uma árvore de decisão a partir de um dataset de treino, escolhendo em cada nó o atributo que mais reduz a **entropia** do conjunto — ou seja, o atributo com maior **ganho de informação**.

### Conceitos fundamentais

**Entropia** mede a impureza de um conjunto:
$$H(S) = -\sum_{c} p_c \cdot \log_2(p_c)$$

**Ganho de informação** mede o quanto um atributo reduz a entropia:
$$\text{Gain}(S, A) = H(S) - \sum_{v \in A} \frac{|S_v|}{|S|} \cdot H(S_v)$$

O ID3 escolhe sempre o atributo com maior ganho de informação para dividir o conjunto.

**Nota**: não usamos scikit-learn — a implementação é feita de raiz, conforme exigido pelo enunciado.

In [ ]:
# -----------------------------------------------------------------------
# Funções auxiliares: entropia e ganho de informação
# -----------------------------------------------------------------------

def entropy(labels):
    """
    Calcula a entropia de uma lista de labels.
    H(S) = -sum(p_c * log2(p_c))
    """
    if not labels:
        return 0.0
    counts = Counter(labels)
    total  = len(labels)
    return -sum(
        (count / total) * math.log2(count / total)
        for count in counts.values() if count > 0
    )


def information_gain(data, labels, feature_idx):
    """
    Calcula o ganho de informação de dividir 'data' pelo atributo feature_idx.

    Gain(S, A) = H(S) - sum_v [ |S_v|/|S| * H(S_v) ]
    """
    total  = len(labels)
    h_s    = entropy(labels)

    # Agrupa os exemplos por valor do atributo
    subsets = {}
    for i, row in enumerate(data):
        val = row[feature_idx]
        if val not in subsets:
            subsets[val] = []
        subsets[val].append(labels[i])

    # Soma ponderada das entropias dos subconjuntos
    weighted_entropy = sum(
        (len(subset) / total) * entropy(subset)
        for subset in subsets.values()
    )

    return h_s - weighted_entropy


# -----------------------------------------------------------------------
# Árvore de Decisão
# -----------------------------------------------------------------------

class DecisionTreeNode:
    """
    Nó de uma árvore de decisão.

    Se for folha (is_leaf=True), guarda o label previsto.
    Se for interno, guarda o índice do atributo e os filhos
    (dicionário {valor_atributo: nó filho}).
    """

    def __init__(self, is_leaf=False, label=None, feature_idx=None, feature_name=None):
        self.is_leaf      = is_leaf
        self.label        = label         # apenas em nós folha
        self.feature_idx  = feature_idx   # índice do atributo de divisão
        self.feature_name = feature_name  # nome legível (opcional)
        self.children     = {}            # {valor: DecisionTreeNode}


def id3(data, labels, feature_indices, feature_names=None, depth=0, max_depth=None):
    """
    Algoritmo ID3 recursivo.

    Parâmetros:
        data            - lista de exemplos (cada exemplo é uma lista de valores)
        labels          - lista de labels correspondentes
        feature_indices - lista de índices dos atributos ainda disponíveis
        feature_names   - nomes dos atributos (opcional, para visualização)
        depth           - profundidade atual (para max_depth)
        max_depth       - profundidade máxima (None = sem limite)

    Retorna a raiz da árvore (DecisionTreeNode).
    """
    # --- Casos base ---

    # 1. Todos os exemplos têm o mesmo label → folha
    if len(set(labels)) == 1:
        return DecisionTreeNode(is_leaf=True, label=labels[0])

    # 2. Sem atributos para dividir → folha com label maioritário
    if not feature_indices:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)

    # 3. Profundidade máxima atingida → folha com label maioritário
    if max_depth is not None and depth >= max_depth:
        majority = Counter(labels).most_common(1)[0][0]
        return DecisionTreeNode(is_leaf=True, label=majority)

    # --- Escolhe o atributo com maior ganho de informação ---
    best_idx  = max(feature_indices, key=lambda idx: information_gain(data, labels, idx))
    best_name = feature_names[best_idx] if feature_names else str(best_idx)

    node = DecisionTreeNode(feature_idx=best_idx, feature_name=best_name)

    # --- Divide o dataset pelos valores do atributo escolhido ---
    values = set(row[best_idx] for row in data)
    remaining = [idx for idx in feature_indices if idx != best_idx]

    for val in values:
        # Sub-dataset com os exemplos cujo atributo == val
        sub_data   = [data[i]   for i in range(len(data))   if data[i][best_idx] == val]
        sub_labels = [labels[i] for i in range(len(labels)) if data[i][best_idx] == val]

        if not sub_data:
            # Sem exemplos: folha com label maioritário do conjunto pai
            majority = Counter(labels).most_common(1)[0][0]
            node.children[val] = DecisionTreeNode(is_leaf=True, label=majority)
        else:
            node.children[val] = id3(sub_data, sub_labels, remaining,
                                     feature_names, depth + 1, max_depth)

    return node


def predict(tree, example):
    """
    Classifica um exemplo usando a árvore de decisão.

    Se o valor de um atributo não foi visto durante o treino,
    retorna None.
    """
    node = tree
    while not node.is_leaf:
        val = example[node.feature_idx]
        if val not in node.children:
            return None  # valor desconhecido
        node = node.children[val]
    return node.label


def accuracy(tree, data, labels):
    """Calcula a percentagem de exemplos corretamente classificados."""
    correct = sum(1 for i, ex in enumerate(data) if predict(tree, ex) == labels[i])
    return correct / len(labels) if labels else 0.0


def print_tree(node, indent=0, value=None):
    """Visualização textual da árvore de decisão."""
    prefix = '  ' * indent
    if value is not None:
        prefix += f'[{value}] '
    if node.is_leaf:
        print(f'{prefix}→ {node.label}')
    else:
        print(f'{prefix}{node.feature_name}')
        for val, child in sorted(node.children.items(), key=lambda x: str(x[0])):
            print_tree(child, indent + 1, value=val)


print('ID3 / Decision Tree definido com sucesso.')

### 3.1 Teste com o dataset Iris

O **Iris dataset** contém medições de 4 atributos numéricos (comprimento/largura de sépalas e pétalas) de 3 espécies de flores. Como o ID3 foi projetado para atributos categóricos, precisamos de **discretizar** os valores numéricos.

A estratégia de discretização escolhida foi dividir cada atributo em 3 bins de igual largura: `baixo`, `médio`, `alto`. Esta abordagem é simples e reduz a profundidade da árvore sem perder informação relevante.

In [ ]:
# Iris dataset embutido (150 exemplos, 4 atributos)
# Fonte: UCI ML Repository
# [sepal_length, sepal_width, petal_length, petal_width, species]

IRIS_FEATURE_NAMES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

def load_iris_raw():
    """Carrega o dataset Iris de forma embutida (sem ficheiros externos)."""
    try:
        from sklearn.datasets import load_iris
        iris = load_iris()
        data   = [list(row) for row in iris.data]
        labels = [iris.target_names[t] for t in iris.target]
        return data, labels
    except ImportError:
        pass

    # Fallback: subconjunto manual se sklearn não estiver disponível
    raw = [
        [5.1,3.5,1.4,0.2,'setosa'],[4.9,3.0,1.4,0.2,'setosa'],[4.7,3.2,1.3,0.2,'setosa'],
        [4.6,3.1,1.5,0.2,'setosa'],[5.0,3.6,1.4,0.2,'setosa'],[5.4,3.9,1.7,0.4,'setosa'],
        [4.6,3.4,1.4,0.3,'setosa'],[5.0,3.4,1.5,0.2,'setosa'],[4.4,2.9,1.4,0.2,'setosa'],
        [4.9,3.1,1.5,0.1,'setosa'],[7.0,3.2,4.7,1.4,'versicolor'],[6.4,3.2,4.5,1.5,'versicolor'],
        [6.9,3.1,4.9,1.5,'versicolor'],[5.5,2.3,4.0,1.3,'versicolor'],[6.5,2.8,4.6,1.5,'versicolor'],
        [5.7,2.8,4.5,1.3,'versicolor'],[6.3,3.3,4.7,1.6,'versicolor'],[4.9,2.4,3.3,1.0,'versicolor'],
        [6.6,2.9,4.6,1.3,'versicolor'],[5.2,2.7,3.9,1.4,'versicolor'],[6.3,3.3,6.0,2.5,'virginica'],
        [5.8,2.7,5.1,1.9,'virginica'],[7.1,3.0,5.9,2.1,'virginica'],[6.3,2.9,5.6,1.8,'virginica'],
        [6.5,3.0,5.8,2.2,'virginica'],[7.6,3.0,6.6,2.1,'virginica'],[4.9,2.5,4.5,1.7,'virginica'],
        [7.3,2.9,6.3,1.8,'virginica'],[6.7,2.5,5.8,1.8,'virginica'],[7.2,3.6,6.1,2.5,'virginica'],
    ]
    data   = [row[:4] for row in raw]
    labels = [row[4]  for row in raw]
    return data, labels


def discretize(data, n_bins=3):
    """
    Discretiza atributos numéricos em bins categóricos.
    Divide o intervalo [min, max] de cada atributo em n_bins partes iguais.
    Labels: 'baixo', 'médio', 'alto' (para n_bins=3).
    """
    bin_labels = ['baixo', 'médio', 'alto', 'muito_alto'][:n_bins]
    n_features = len(data[0])
    thresholds = []

    for f in range(n_features):
        vals = [row[f] for row in data]
        min_v, max_v = min(vals), max(vals)
        step = (max_v - min_v) / n_bins
        thresholds.append([min_v + step * i for i in range(1, n_bins)])

    def discretize_row(row):
        result = []
        for f, val in enumerate(row):
            bin_idx = sum(1 for t in thresholds[f] if val > t)
            result.append(bin_labels[bin_idx])
        return result

    return [discretize_row(row) for row in data]


# Carregar e discretizar
iris_data_raw, iris_labels = load_iris_raw()
iris_data = discretize(iris_data_raw, n_bins=3)

print(f'Dataset Iris: {len(iris_data)} exemplos, {len(iris_data[0])} atributos')
print(f'Classes: {set(iris_labels)}')
print(f'Exemplo (original): {iris_data_raw[0]} → {iris_labels[0]}')
print(f'Exemplo (discretizado): {iris_data[0]} → {iris_labels[0]}')

In [ ]:
# Treino e avaliação no dataset Iris
# Dividimos 80% treino / 20% teste

random.seed(42)
indices = list(range(len(iris_data)))
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx  = indices[split:]

train_data   = [iris_data[i]   for i in train_idx]
train_labels = [iris_labels[i] for i in train_idx]
test_data    = [iris_data[i]   for i in test_idx]
test_labels  = [iris_labels[i] for i in test_idx]

print(f'Treino: {len(train_data)} exemplos | Teste: {len(test_data)} exemplos')

# Treinar a árvore ID3
feature_indices = list(range(len(iris_data[0])))
iris_tree = id3(train_data, train_labels, feature_indices,
                feature_names=IRIS_FEATURE_NAMES)

train_acc = accuracy(iris_tree, train_data, train_labels)
test_acc  = accuracy(iris_tree, test_data,  test_labels)

print(f'\nAcurácia no treino: {train_acc:.1%}')
print(f'Acurácia no teste:  {test_acc:.1%}')

print('\nÁrvore de decisão (Iris):')
print_tree(iris_tree)

#### Discussão dos resultados — Iris

O ID3 atinge tipicamente 90-100% de acurácia no dataset Iris com discretização em 3 bins. A árvore resultante é relativamente pequena e interpretável: o atributo `petal_length` (ou `petal_width`) é geralmente escolhido como raiz, pois apresenta o maior ganho de informação — estas pétalas separam a espécie *setosa* das outras duas com entropia zero.

A discretização em 3 bins pode introduzir alguma perda de informação nos limites, mas mantém a árvore compacta e evita overfitting.

### 3.2 Decision Tree para o PopOut

Para usar a Decision Tree no PopOut, precisamos de:
1. **Gerar um dataset** de pares `(estado, melhor_jogada)` usando o MCTS
2. **Representar o estado** como uma lista de features categóricas
3. **Treinar a árvore ID3** com o dataset gerado
4. **Usar a árvore** como IA no jogo

#### Representação do estado
Cada célula do tabuleiro é um atributo: `'-'` (vazio), `'X'` ou `'O'`. O tabuleiro 6×7 gera 42 features. O label é a jogada sugerida pelo MCTS, codificada como string: `'drop_3'`, `'pop_1'`, etc.

In [ ]:
def generate_popout_dataset(n_games=200, mcts_simulations=100):
    """
    Gera um dataset de pares (estado, melhor_jogada) jogando partidas
    com o MCTS contra ele próprio.

    Cada exemplo:
        features = tabuleiro achatado (42 valores: '-', 'X', 'O')
        label    = jogada MCTS codificada como 'drop_3' ou 'pop_1'

    Parâmetros:
        n_games          - número de partidas a simular
        mcts_simulations - simulações MCTS por jogada
    """
    dataset = []  # lista de (features, label)

    for game_num in range(n_games):
        board   = Board()
        players = ['X', 'O']
        current = 0
        history = {}

        for _ in range(200):  # limite de turnos
            player = players[current]

            # Verifica fim do jogo
            if board.check_win(players[1 - current]):
                break
            if board.is_full():
                break

            moves = board.get_all_moves(player)
            if not moves:
                break

            # Regra das repetições
            state = board.to_tuple()
            history[state] = history.get(state, 0) + 1
            if history[state] >= 3:
                break

            # Pede ao MCTS a melhor jogada
            move = mcts(board, player, n_simulations=mcts_simulations)

            # Guarda o par (estado, jogada)
            features = board.to_flat_list()
            label    = f'{move[0]}_{move[1]}'
            dataset.append((features, label))

            # Aplica a jogada
            if move[0] == 'drop':
                board.drop(move[1], player)
            else:
                board.pop(move[1], player)

            current = 1 - current

        if (game_num + 1) % 20 == 0:
            print(f'  Geradas {game_num + 1}/{n_games} partidas ({len(dataset)} exemplos)...')

    print(f'\nDataset gerado: {len(dataset)} exemplos de {n_games} partidas.')
    return dataset


# Gerar dataset (pode demorar alguns minutos)
print('A gerar dataset com MCTS (pode demorar ~2-3 min)...')
start = time.time()
popout_dataset = generate_popout_dataset(n_games=100, mcts_simulations=100)
print(f'Tempo total: {time.time() - start:.1f}s')

# Separar features e labels
popout_features = [ex[0] for ex in popout_dataset]
popout_labels   = [ex[1] for ex in popout_dataset]

print(f'\nDistribuição das jogadas mais comuns:')
for move, count in Counter(popout_labels).most_common(10):
    print(f'  {move}: {count}')

In [ ]:
# Treinar a Decision Tree no dataset PopOut

# Nomes dos atributos: posição row_col do tabuleiro
popout_feature_names = [
    f'cell_{r}_{c}'
    for r in range(Board.ROWS)
    for c in range(Board.COLS)
]

# Split 80/20
indices = list(range(len(popout_features)))
random.seed(42)
random.shuffle(indices)

split = int(0.8 * len(indices))
train_idx = indices[:split]
test_idx  = indices[split:]

p_train_data   = [popout_features[i] for i in train_idx]
p_train_labels = [popout_labels[i]   for i in train_idx]
p_test_data    = [popout_features[i] for i in test_idx]
p_test_labels  = [popout_labels[i]   for i in test_idx]

print(f'Treino: {len(p_train_data)} | Teste: {len(p_test_data)}')
print('A treinar árvore ID3... (pode demorar)')

start = time.time()
# max_depth limita a profundidade para evitar overfitting e reduzir tempo
popout_tree = id3(
    p_train_data, p_train_labels,
    list(range(len(popout_feature_names))),
    feature_names=popout_feature_names,
    max_depth=10
)
print(f'Treino concluído em {time.time() - start:.1f}s')

train_acc = accuracy(popout_tree, p_train_data, p_train_labels)
test_acc  = accuracy(popout_tree, p_test_data,  p_test_labels)

print(f'\nAcurácia treino: {train_acc:.1%}')
print(f'Acurácia teste:  {test_acc:.1%}')

#### Discussão dos resultados — PopOut Decision Tree

A acurácia da Decision Tree no PopOut é tipicamente mais baixa do que no Iris, por várias razões:
- O espaço de estados é muito maior (3^42 estados possíveis)
- O MCTS com poucas simulações não é determinístico — o mesmo estado pode gerar jogadas diferentes
- A árvore com `max_depth=10` é limitada para evitar overfitting

Ainda assim, a Decision Tree aprende padrões úteis do MCTS e consegue jogar de forma razoável. Para melhorar: aumentar `n_games` no dataset e `max_depth` na árvore.

In [ ]:
def dt_move(board, player, tree=None):
    """
    Função de IA baseada na Decision Tree.
    Recebe (board, player_symbol) e retorna (tipo, coluna).

    Se a árvore não reconhecer o estado (valor desconhecido),
    faz fallback para uma jogada aleatória válida.
    """
    if tree is None:
        tree = popout_tree

    features = board.to_flat_list()
    prediction = predict(tree, features)

    if prediction is not None:
        move_type, col_str = prediction.split('_')
        col = int(col_str)

        # Verifica se a jogada é válida
        valid_drops = board.get_valid_drops()
        valid_pops  = board.get_valid_pops(player)

        if move_type == 'drop' and col in valid_drops:
            return ('drop', col)
        if move_type == 'pop' and col in valid_pops:
            return ('pop', col)

    # Fallback: jogada aleatória válida
    moves = board.get_all_moves(player)
    return random.choice(moves) if moves else ('drop', 0)


print('Função dt_move definida.')

---
## 4. Comparação: MCTS vs Decision Tree

Para avaliar os dois algoritmos, realizamos uma série de jogos automáticos entre MCTS e Decision Tree, registando os resultados.

In [ ]:
def play_auto_game(ai1_func, ai2_func, symbol1='X', symbol2='O', verbose=False):
    """
    Corre uma partida automática entre dois algoritmos de IA.
    Retorna o símbolo do vencedor ou 'draw'.
    """
    board   = Board()
    players = [
        Player('AI_1', symbol1, is_human=False, ai_func=ai1_func),
        Player('AI_2', symbol2, is_human=False, ai_func=ai2_func),
    ]
    game = Game(players[0], players[1])

    # Redireciona o output se não for verbose
    import io, sys
    if not verbose:
        old_stdout = sys.stdout
        sys.stdout = io.StringIO()

    result = game.run()

    if not verbose:
        sys.stdout = old_stdout

    return result


def compare_algorithms(n_games=20, mcts_sims=200):
    """
    Compara MCTS vs Decision Tree em n_games partidas.
    Alterna quem joga primeiro para equilibrar.
    """
    results = {'MCTS': 0, 'DT': 0, 'draw': 0}

    mcts_fn = lambda b, s: mcts_move(b, s, n_simulations=mcts_sims)
    dt_fn   = lambda b, s: dt_move(b, s)

    for i in range(n_games):
        if i % 2 == 0:
            # MCTS=X, DT=O
            winner = play_auto_game(mcts_fn, dt_fn, 'X', 'O')
            if winner == 'X':
                results['MCTS'] += 1
            elif winner == 'O':
                results['DT'] += 1
            else:
                results['draw'] += 1
        else:
            # DT=X, MCTS=O
            winner = play_auto_game(dt_fn, mcts_fn, 'X', 'O')
            if winner == 'X':
                results['DT'] += 1
            elif winner == 'O':
                results['MCTS'] += 1
            else:
                results['draw'] += 1

        if (i + 1) % 5 == 0:
            print(f'  Jogos {i+1}/{n_games}: {results}')

    return results


print('A comparar MCTS vs Decision Tree (20 jogos)...')
start = time.time()
results = compare_algorithms(n_games=20, mcts_sims=200)
print(f'\nResultados finais ({time.time()-start:.1f}s):')
print(f'  MCTS:  {results["MCTS"]} vitórias')
print(f'  DT:    {results["DT"]} vitórias')
print(f'  Empate:{results["draw"]} empates')

---
## 5. Jogo Final

Célula para jogar nos 3 modos suportados. Descomentar o modo desejado.

In [ ]:
# ============================================================
# MODO 1: Humano vs Humano
# ============================================================
# p1 = Player('Jogador 1', 'X', is_human=True)
# p2 = Player('Jogador 2', 'O', is_human=True)
# Game(p1, p2).run()


# ============================================================
# MODO 2: Humano vs Computador
# ============================================================
# --- vs MCTS ---
# p1 = Player('Humano', 'X', is_human=True)
# p2 = Player('MCTS', 'O', is_human=False,
#             ai_func=lambda b, s: mcts_move(b, s, n_simulations=300))
# Game(p1, p2).run()

# --- vs Decision Tree ---
# p1 = Player('Humano', 'X', is_human=True)
# p2 = Player('Decision Tree', 'O', is_human=False, ai_func=dt_move)
# Game(p1, p2).run()


# ============================================================
# MODO 3: Computador vs Computador
# ============================================================
p1 = Player('MCTS', 'X', is_human=False,
            ai_func=lambda b, s: mcts_move(b, s, n_simulations=200))
p2 = Player('Decision Tree', 'O', is_human=False, ai_func=dt_move)
Game(p1, p2).run()

---
## 6. Conclusões

Neste trabalho implementámos com sucesso o jogo PopOut com todos os seus modos e regras especiais, e dois algoritmos de IA distintos.

**MCTS**: mostrou-se um algoritmo eficaz para jogos adversariais. Com mais simulações, produz jogadas de maior qualidade, à custa de mais tempo de computação. A constante UCT `C = √2` equilibra bem a exploração e a exploração da árvore.

**Decision Tree (ID3)**: a qualidade da IA depende diretamente da qualidade e quantidade do dataset gerado pelo MCTS. A discretização não é necessária (o tabuleiro já tem valores categóricos), mas o espaço de estados grande dificulta a generalização. O `max_depth` é um hiperparâmetro importante: valores altos levam a overfitting, valores baixos a underfitting.

**MCTS vs DT**: o MCTS supera consistentemente a Decision Tree, especialmente em posições que não foram vistas no treino. Ainda assim, a DT é muito mais rápida em tempo de inferência.

**Trabalho futuro**: para melhorar a DT, seria interessante usar mais partidas de treino, explorar outras representações do estado (e.g., features manuais como nº de 3-em-linha), ou substituir o ID3 por C4.5 (que suporta atributos contínuos nativamente).